# Acoustic Feature Extraction Pipeline

> **Note on Reproducibility:** *This notebook instantiates the acoustic feature extraction pipeline for the IDSM dataset. It leverages high-performance parallel processing to segment the raw `.wav` rainforest recordings and compute domain-specific acoustic metrics (e.g., PSD, ZCR, MFCCs). The final outputs are structured CSV files (`*_metrics.csv`) which serve as the foundation for the Machine Learning experiments.*

In [1]:
import sys
from pathlib import Path

current_path = Path.cwd()
PROJECT_ROOT = None

for p in [current_path, current_path.parent, current_path.parent.parent]:
    if (p / "rainfall_acoustic_classification").exists():
        PROJECT_ROOT = p
        break

if PROJECT_ROOT:
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT)) # insert(0) garante prioridade absoluta
    print(f"✅ Raiz do projeto adicionada ao sys.path: {PROJECT_ROOT}")
else:
    print("❌ ERRO: Não foi possível encontrar a raiz do projeto.")

✅ Raiz do projeto adicionada ao sys.path: c:\Users\ggrmi\Documents\SI-Tarcisio\Sound-of-Rainfall\RAC-semiurban-forest-ML-2026


## Parallel Pipeline Configuration
*Instantiation of the dedicated audio processing worker framework. This engine encapsulates the signal conditioning, target sample rate adjustment (24 kHz), and mathematical feature extraction blocks, allowing parallel execution across multiple CPU cores via `joblib`.*

In [2]:
%%writefile IDSM_pipeline.py
import sys
import gc
from pathlib import Path
from typing import Dict, Any, List

# ====================================================================
# 1. PATH INJECTION FOR WORKERS
# Child processes must discover the project root independently.
# ====================================================================
current_path = Path.cwd()
for p in [current_path, current_path.parent, current_path.parent.parent]:
    if (p / "rainfall_acoustic_classification").exists():
        if str(p) not in sys.path:
            sys.path.insert(0, str(p))
        break

# ====================================================================
# 2. LIBRARY IMPORTS AND CONFIGURATION
# ====================================================================
from rainfall_acoustic_classification.core import load_audio_sample
from rainfall_acoustic_classification.processing import (
    AudioAugmenter, AudioSegmenter, AcousticMetrics
)

augmenter = AudioAugmenter(sample_rate=24000, noise_prob=0.2, cutout_prob=0.3, random_state=42)
segmenter = AudioSegmenter(sample_rate=24000,segment_duration=10.0, overlap=0.5)
metrics_ext = AcousticMetrics(sample_rate=24000, fft_window_size=1024)

def audio_processing_worker(row_dict: Dict[str, Any]) -> List[Dict[str, Any]]:
    """
    Worker function to execute signal processing and feature extraction 
    for a single audio file.

    This function loads the audio, applies stochastic augmentation (if flagged),
    segments the signal, and extracts acoustic metrics for each segment.

    Parameters
    ----------
    row_dict : dict
        A dictionary representing a single row from the metadata DataFrame. Must contain
        at least the 'file_path', 'split', and 'should_augment' keys.

    Returns
    -------
    list of dict
        A list of dictionaries, where each dictionary contains the extracted features
        and metadata for a specific segment of the original audio file. Returns an
        empty list if the audio loading fails.
    """
    sample = load_audio_sample(row_dict.get('file_path'), sample_rate=24000)
    if not sample: 
        return []
    
    y = sample.audio_data
    aug_log = "raw"
    
    # Apply stochastic augmentation strictly if the flag is True
    if row_dict.get('split') == 'train' and row_dict.get('should_augment', False):
        y, aug_log = augmenter.process(y)
        
    results = []
    chunks = segmenter.process(y)
    
    for idx, (chunk_array, offset) in enumerate(chunks):
        features = metrics_ext.calculate(chunk_array)
        
        segment_data = row_dict.copy()
        segment_data.update({
            'segment_idx': idx, 
            'offset_sec': offset, 
            'aug_params': aug_log
        })
        segment_data.update(features)
        results.append(segment_data)
        
    # =======================================================
    # AGGRESSIVE GARBAGE COLLECTION
    # =======================================================
    del y
    del chunks
    del sample
    gc.collect() 
    
    return results

Overwriting IDSM_pipeline.py


## Signal Processing and Feature Extraction
*Execution of the parallel feature extraction pipeline across the IDSM data splits. The continuous environmental audio streams are framed into discrete segments, transforming raw wave signals into structured feature matrices.*

In [3]:
import pandas as pd
from rainfall_acoustic_classification.utils import parallel_pipe
from IDSM_pipeline import audio_processing_worker
from rainfall_acoustic_classification.utils import prepare_balanced_training_set

split_file_path = PROJECT_ROOT / "data" / "splits" / "IDSM"
metrics_file_path = PROJECT_ROOT / "data" / "processed" / "IDSM"

### IDSM Training Set Processing and Augmentation
*Processing the training partition. Stochastic data augmentation (such as background noise injection and adaptive cutout techniques) is strictly restricted to this phase to artificially balance minority classes and enhance classifier robustness.*

In [4]:
train_file_path = split_file_path / "IDSM_train.csv"

# Load and tag the split
df_train_meta = pd.read_csv(train_file_path)
df_train_meta['split'] = 'train'

# Define the rain classes according to the methodology
RAIN_CLASSES = ['light', 'moderate', 'heavy', 'violent']

# Apply the balancing function
df_train_final = prepare_balanced_training_set(
    df_train=df_train_meta, 
    rain_classes=RAIN_CLASSES, 
    random_state=42
)

print(f"\n Total tasks dispatched to workers: {len(df_train_final)} files.")


📊 Intra-Class Balancing Strategy:
   -> Majority Class: 'moderate' with N_max = 407 samples.
   -> [light] Originals: 352 | Augmented generated: +55 | Final Total: 407
   -> [moderate] Originals: 407 | Already at N_max. No augmentation applied.
   -> [heavy] Originals: 149 | Augmented generated: +258 | Final Total: 407
   -> [violent] Originals: 40 | Augmented generated: +367 | Final Total: 407

 Total tasks dispatched to workers: 2576 files.


In [5]:
# Execute the parallel pipeline
df_train_final_features = df_train_final.pipe(
    parallel_pipe, 
    worker_func=audio_processing_worker, 
    n_jobs=20, 
    desc="Extracting IDSM Features"
)

Extracting IDSM Features:   0%|          | 0/2576 [00:00<?, ?file/s]

In [6]:
train_metrics_file_path = metrics_file_path / "IDSM_train_metrics.csv"
df_train_final_features.to_csv(train_metrics_file_path, index=False)

### Validation and Test Sets Processing
*Processing the evaluation partitions of the IDSM dataset. Data augmentation is completely bypassed for the Validation and Test sets to preserve the authentic acoustic environment of the forest, ensuring an unbiased model performance verification.*

In [7]:
val_file_path = split_file_path / "IDSM_val.csv"

df_val_meta = pd.read_csv(val_file_path)
df_val_meta['split'] = 'val'

df_val_final_features = df_val_meta.pipe(
    parallel_pipe, 
    worker_func=audio_processing_worker, 
    n_jobs=20, 
    desc="Extraindo Features IDSM"
)

Extraindo Features IDSM:   0%|          | 0/238 [00:00<?, ?file/s]

In [8]:
val_metrics_file_path = metrics_file_path / "IDSM_val_metrics.csv"
df_val_final_features.to_csv(val_metrics_file_path, index=False)

In [9]:
test_file_path = split_file_path / "IDSM_test.csv"

df_test_meta = pd.read_csv(test_file_path)
df_test_meta['split'] = 'test'

df_test_final_features = df_test_meta.pipe(
    parallel_pipe, 
    worker_func=audio_processing_worker, 
    n_jobs=20, 
    desc="Extraindo Features IDSM"
)

Extraindo Features IDSM:   0%|          | 0/238 [00:00<?, ?file/s]

In [10]:
test_metrics_file_path = metrics_file_path / "IDSM_test_metrics.csv"
df_test_final_features.to_csv(test_metrics_file_path, index=False)

In [11]:
count_train = df_train_final_features['category'].value_counts()
print('TRAIN:')
print(count_train)
print("Total Samples    ", len(df_train_final_features['category']), '\n')

count_val = df_val_final_features['category'].value_counts()
print('VALIDATION:')
print(count_val)
print("Total Samples    ", len(df_val_final_features['category']), '\n')

count_test = df_test_final_features['category'].value_counts()
print('TEST')
print(count_test)
print("Total Samples    ", len(df_test_final_features['category']), '\n')

total = len(df_test_final_features['category']) + len(df_val_final_features['category']) + len(df_train_final_features['category'])

print("PERCENTAGES:")
print("Train         ", round((len(df_train_final_features['category'])/total), 2))
print("Validation    ", round((len(df_val_final_features['category'])/total), 2))
print("Test          ", round((len(df_test_final_features['category'])/total), 2))

TRAIN:
category
no-rain     10423
moderate     4477
heavy        4477
light        4477
violent      4477
Name: count, dtype: int64
Total Samples     28331 

VALIDATION:
category
no-rain     1309
moderate     561
light        484
heavy        209
violent       55
Name: count, dtype: int64
Total Samples     2618 

TEST
category
no-rain     1309
moderate     561
light        484
heavy        209
violent       55
Name: count, dtype: int64
Total Samples     2618 

PERCENTAGES:
Train          0.84
Validation     0.08
Test           0.08


In [12]:
df_train_final_features.groupby(['category']).sample()

,file_name,timestamp,period,mm_5min,mm_hr,category,recorder,location,file_path,extension,...,wav_detail_lvl1_energy,wav_detail_lvl1_std,wav_energy_mean,roughness,tfsd,wav_detail_lvl1_var,wav_detail_lvl2_var,wav_detail_lvl3_var,wav_detail_lvl4_var,wav_detail_lvl5_var
22447,SMM00894_20230301_133500_1-6_heavy_afternoon.wav,2023-03-01 13:35:00,afternoon,1.6,19.2,heavy,SMM00894,IDSM,C:\Users\ggrmi\Documents\SI-Tarcisio\Sound-of-...,.wav,...,0.000026,0.005078,0.013974,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5625,SMM00894_20230312_131000_0-2_light_afternoon.wav,2023-03-12 13:10:00,afternoon,0.2,2.4,light,SMM00894,IDSM,C:\Users\ggrmi\Documents\SI-Tarcisio\Sound-of-...,.wav,...,0.000001,0.001151,0.008663,NaN,NaN,NaN,NaN,NaN,NaN,NaN
19502,SMM00894_20230411_033000_0-4_moderate_night.wav,2023-04-11 03:30:00,overnight,0.4,4.8,moderate,SMM00894,IDSM,C:\Users\ggrmi\Documents\SI-Tarcisio\Sound-of-...,.wav,...,0.001737,0.041671,0.006451,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9109,SMM00894_20230413_151000_0-0_no-rain_afternoon...,2023-04-13 15:10:00,afternoon,0.0,0.0,no-rain,SMM00894,IDSM,C:\Users\ggrmi\Documents\SI-Tarcisio\Sound-of-...,.wav,...,0.000099,0.009944,0.000084,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27101,SMM00894_20230319_193000_6-4_violent_night.wav,2023-03-19 19:30:00,night,6.4,76.8,violent,SMM00894,IDSM,C:\Users\ggrmi\Documents\SI-Tarcisio\Sound-of-...,.wav,...,0.000327,0.018092,0.150184,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
df_val_final_features.groupby(['category']).sample()

,file_name,timestamp,period,mm_5min,mm_hr,category,recorder,location,file_path,extension,...,wav_detail_lvl1_energy,wav_detail_lvl1_std,wav_energy_mean,roughness,tfsd,wav_detail_lvl1_var,wav_detail_lvl2_var,wav_detail_lvl3_var,wav_detail_lvl4_var,wav_detail_lvl5_var
2195,SMM00894_20230220_144000_2-6_heavy_afternoon.wav,2023-02-20 14:40:00,afternoon,2.6,31.2,heavy,SMM00894,IDSM,C:\Users\ggrmi\Documents\SI-Tarcisio\Sound-of-...,.wav,...,0.000106,0.010304,0.023521,NaN,NaN,NaN,NaN,NaN,NaN,NaN
909,SMM00894_20230217_000500_0-2_light_night.wav,2023-02-17 00:05:00,overnight,0.2,2.4,light,SMM00894,IDSM,C:\Users\ggrmi\Documents\SI-Tarcisio\Sound-of-...,.wav,...,0.001167,0.034164,0.011538,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2399,SMM00894_20230407_221500_0-8_moderate_night.wav,2023-04-07 22:15:00,night,0.8,9.6,moderate,SMM00894,IDSM,C:\Users\ggrmi\Documents\SI-Tarcisio\Sound-of-...,.wav,...,0.008735,0.093459,0.048152,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1066,SMM00894_20230222_223500_0-0_no-rain_night.wav,2023-02-22 22:35:00,night,0.0,0.0,no-rain,SMM00894,IDSM,C:\Users\ggrmi\Documents\SI-Tarcisio\Sound-of-...,.wav,...,0.003638,0.060314,0.001251,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2291,SMM00894_20230421_133000_10-0_violent_afternoo...,2023-04-21 13:30:00,afternoon,10.0,120.0,violent,SMM00894,IDSM,C:\Users\ggrmi\Documents\SI-Tarcisio\Sound-of-...,.wav,...,0.000106,0.010299,0.135881,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
df_test_final_features.groupby(['category']).sample()

,file_name,timestamp,period,mm_5min,mm_hr,category,recorder,location,file_path,extension,...,wav_detail_lvl1_energy,wav_detail_lvl1_std,wav_energy_mean,roughness,tfsd,wav_detail_lvl1_var,wav_detail_lvl2_var,wav_detail_lvl3_var,wav_detail_lvl4_var,wav_detail_lvl5_var
1082,SMM00894_20230512_140500_2-4_heavy_afternoon.wav,2023-05-12 14:05:00,afternoon,2.4,28.8,heavy,SMM00894,IDSM,C:\Users\ggrmi\Documents\SI-Tarcisio\Sound-of-...,.wav,...,0.000049,0.007002,0.066711,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1547,SMM00894_20230413_230000_0-2_light_night.wav,2023-04-13 23:00:00,night,0.2,2.4,light,SMM00894,IDSM,C:\Users\ggrmi\Documents\SI-Tarcisio\Sound-of-...,.wav,...,0.017442,0.132067,0.074852,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1130,SMM00894_20230506_061000_0-8_moderate_morning.wav,2023-05-06 06:10:00,morning,0.8,9.6,moderate,SMM00894,IDSM,C:\Users\ggrmi\Documents\SI-Tarcisio\Sound-of-...,.wav,...,0.000003,0.001748,0.015282,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2582,SMM00894_20230512_055500_0-0_no-rain_night.wav,2023-05-12 05:55:00,overnight,0.0,0.0,no-rain,SMM00894,IDSM,C:\Users\ggrmi\Documents\SI-Tarcisio\Sound-of-...,.wav,...,0.000132,0.011511,0.000067,NaN,NaN,NaN,NaN,NaN,NaN,NaN
927,SMM00894_20230424_122500_5-0_violent_afternoon...,2023-04-24 12:25:00,afternoon,5.0,60.0,violent,SMM00894,IDSM,C:\Users\ggrmi\Documents\SI-Tarcisio\Sound-of-...,.wav,...,0.001020,0.031931,0.264503,NaN,NaN,NaN,NaN,NaN,NaN,NaN
